In [13]:
import os
import timm
import json
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt


In [ ]:
CONFIG = {
    'BS': 16,
    'LR': 1e-5,
    'WD': 1e-4,
    'EPOCHS': 80,
    'WORKERS': 8,
    'DEVICE': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'pretrained': True,
    'num_classes': 5,
    'tile_size': 256,
}

DATA_DIR = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/classif_256x256_newannot"
OUTPUT_DIR = "models/classifier/"
version = '1'
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Resize(256,256,cv2.INTER_LINEAR),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    A.Resize(256,256,cv2.INTER_LINEAR),
    ToTensorV2()
])


class ClassificationDataset(Dataset):
    def __init__(self, image_paths, label_paths, transform=None, num_classes=5):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.transform = transform
        self.num_classes = num_classes
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        with open(self.label_paths[idx], 'r') as f:
            label_data = json.load(f)
        
        classes_present = label_data['classes']
        
        label = torch.tensor(classes_present, dtype=torch.float32)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, label


def get_data_paths(split):
    split_dir = os.path.join(DATA_DIR, split)
    images_dir = os.path.join(split_dir, "images")
    labels_dir = os.path.join(split_dir, "labels")
    
    image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
    
    image_paths = [os.path.join(images_dir, f) for f in image_files]
    label_paths = [os.path.join(labels_dir, f.replace('.jpg', '.json')) for f in image_files]
    
    return image_paths, label_paths


def create_model(backbone='resnet152'):
    import timm

    model = timm.create_model('resnet152', 
                              pretrained=CONFIG['pretrained'], 
                              num_classes=CONFIG['num_classes'])
    return model


def calculate_class_weights(label_paths, num_classes=5):
    class_counts = np.zeros(num_classes)
    
    num_samples = min(len(label_paths), 2000)
    for label_path in tqdm(label_paths[:num_samples], desc="Calculating class weights"):
        with open(label_path, 'r') as f:
            label_data = json.load(f)
        classes = label_data['classes']
        for i, c in enumerate(classes):
            class_counts[i] += c
    
    pos_weight = (num_samples - class_counts) / (class_counts + 1e-6)
    
    print(f"Class counts: {class_counts}")
    print(f"Pos weight (neg/pos): {pos_weight}")
    
    return torch.tensor(pos_weight, dtype=torch.float32)


class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
    
    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
        
        return self.counter >= self.patience


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    return running_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item() / labels.shape[1]
            total += labels.size(0)
    
    accuracy = correct / total
    return running_loss / len(loader), accuracy


def main():
    print(f"Device: {CONFIG['DEVICE']}")
    print(f"Backbone: resnet152")
    print(f"Num classes: {CONFIG['num_classes']}")
    
    train_images, train_labels = get_data_paths('train')
    val_images, val_labels = get_data_paths('val')
    test_images, test_labels = get_data_paths('test')
    
    print(f"Train: {len(train_images)}, Val: {len(val_images)}, Test: {len(test_images)}")
    
    train_dataset = ClassificationDataset(train_images, train_labels, transform=train_transform)
    val_dataset = ClassificationDataset(val_images, val_labels, transform=valid_transform)
    test_dataset = ClassificationDataset(test_images, test_labels, transform=valid_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['BS'], shuffle=True, 
                             num_workers=CONFIG['WORKERS'], pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['BS'], shuffle=False,
                           num_workers=CONFIG['WORKERS'], pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['BS'], shuffle=False,
                            num_workers=CONFIG['WORKERS'], pin_memory=True)
    
    class_weights = calculate_class_weights(train_labels, CONFIG['num_classes'])
    
    print(f"Class weights (before GPU): {class_weights}")
    print(f"Has NaN: {torch.isnan(class_weights).any()}")
    print(f"Has Inf: {torch.isinf(class_weights).any()}")
    
    class_weights = class_weights.to(CONFIG['DEVICE'])

    model = create_model('resnet152')
    model = model.to(CONFIG['DEVICE'])
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['LR'], weight_decay=CONFIG['WD'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['EPOCHS'])
    early_stopping = EarlyStopping(patience=10, min_delta=0.001)
    
    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    print("\nStarting training...")
    for epoch in range(CONFIG['EPOCHS']):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, CONFIG['DEVICE'])
        val_loss, val_acc = validate(model, val_loader, criterion, CONFIG['DEVICE'])
        
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{CONFIG['EPOCHS']} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'config': CONFIG
            }, OUTPUT_DIR + f'{version}/' + 'unet_resnet152_classification_best.pth')
            print(f"  -> Best model saved! (Acc: {val_acc:.4f})")
        
        if early_stopping.step(val_acc):
            print("Early stopping triggered!")
            break
    
    print("\nTraining completed!")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    
    plt.figure(figsize=(10, 4))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.legend()
    plt.title('Training History')
    plt.savefig(OUTPUT_DIR + f'{version}/' + 'training_history.png')
    plt.show()


if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR + f'{version}/', exist_ok=True)
    main()


/tmp/ipykernel_7391/1331896899.py:23: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


Device: cpu
Backbone: resnet152
Num classes: 5
Train: 46353, Val: 5424, Test: 5235


Calculating class weights: 100%|██████████| 2000/2000 [00:00<00:00, 60104.95it/s]


Class counts: [ 223.  465.  323.  406. 2000.]
Pos weight (neg/pos): [7.96860983 3.30107526 5.19195045 3.92610836 0.        ]
Class weights (before GPU): tensor([7.9686, 3.3011, 5.1920, 3.9261, 0.0000])
Has NaN: False
Has Inf: False

Starting training...


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
